# Chặng 8 và 12 — Phát hiện botnet và nhận diện topology

Dữ liệu: CTU-13 (13 kịch bản botnet thật), chia theo kịch bản. Đặc trưng đồ thị liên lạc và gọi định kỳ
(beaconing) trên host–cửa sổ 120 giây. Hai mô hình: Logistic Regression và Random Forest. Topology tập trung/ngang
hàng theo quy tắc. Giải thích: `docs/45`–`52`, lab: `docs/73`, `76`, `80`.

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
F = ROOT / "output" / "features"
H = ROOT / "docs" / "hinh"

from nids import botnet as bn
df = bn.load_labeled(ROOT / "output" / "ctu13", sorted(bn.SCENARIOS))
pd.crosstab(df["split"], df[bn.LABEL_COL].fillna("bo"))

label_host,background,bo,normal_host,suspect_host
split,,,,
test,148720,6489,24880,19188
train,183958,11977,32827,9575
validation,60131,1231,10453,6555


## 1. Hai mô hình phát hiện host bot trên test CTU-13

In [2]:
rows = {}
for name in ("botnet_lr", "botnet_rf"):
    m = bn.load_model(ROOT / "models" / f"{name}.joblib")
    r = bn.evaluate(m, df, "test")
    rows[name] = {**r["tong"], "background_bi_bao": r["background_bi_bao"], "nguong": m.threshold}
pd.DataFrame(rows).T

,precision,recall,f1,ty_le_bao_nham,tp,fp,fn,tn,background_bi_bao,nguong
botnet_lr,0.9490,0.1319,0.2315,0.00547,2530.0,136.0,16658.0,24744.0,0.1069,0.3
botnet_rf,0.9706,0.6775,0.7979,0.01584,12999.0,394.0,6189.0,24486.0,0.1193,0.1


## 2. Đặc trưng quan trọng của Random Forest: gọi định kỳ và đồ thị

In [3]:
pd.Series(bn.importance(bn.load_model(ROOT / "models" / "botnet_rf.joblib"), top=12))

so_peer                        0.1739
tap_trung_ra                   0.1482
ty_le_khong_tra_loi_ra         0.1085
so_peer_ra                     0.0983
so_byte_ra                     0.0843
so_goi_ra                      0.0578
so_flow_ra                     0.0451
so_flow_vao                    0.0436
entropy_peer_ra                0.0371
ty_le_udp                      0.0289
beacon_so_quan_sat_lon_nhat    0.0263
so_cong_ra                     0.0191
dtype: float64

## 3. Beaconing: bot gọi đều, máy bình thường thì không

In [4]:
tr = df[(df["split"] == "train") & df[bn.LABEL_COL].isin([bn.SUSPECT, bn.NORMAL])]
tr.groupby(bn.LABEL_COL)[["beacon_so_cap_deu", "beacon_cv_nho_nhat", "beacon_chu_ky", "so_peer",
                          "tap_trung_ra", "ty_le_khong_tra_loi_ra"]].median().T

label_host,normal_host,suspect_host
beacon_so_cap_deu,1.000000,0.000000
beacon_cv_nho_nhat,0.015350,1.676026
beacon_chu_ky,30.527225,3.336719
so_peer,8.000000,58.000000
tap_trung_ra,0.666667,0.017857
ty_le_khong_tra_loi_ra,0.000000,0.966667


## 4. Lab: năm bot, topology tập trung và ngang hàng, đối chứng lành tính

In [5]:
for f in ("bot_lab3.json", "bot_b2.json"):
    s = json.loads((H / "thuc_nghiem" / "lab" / "bo3" / f).read_text(encoding="utf-8"))["tong_hop"]
    print(f)
    display(pd.DataFrame(s).T[["so_lan_chay", "bot_bi_bao", "tong_bot", "ty_le_topology", "cua_so_bao_nham"]])

bot_lab3.json


,so_lan_chay,bot_bi_bao,tong_bot,ty_le_topology,cua_so_bao_nham
k4-bien-thien,3,9,9,1.0,0
k4-co-dinh,3,9,9,1.0,0
k4-doi-chung,3,NaN,NaN,NaN,0
k5-doi-chung,3,NaN,NaN,NaN,0
k5-p2p,3,9,9,0.99,0
k6-doi-chung,3,NaN,NaN,NaN,0
k6-p2p,3,15,15,0.992,0
k6-tap-trung,3,15,15,1.0,0
x4-doi-chung-doi-vai,3,NaN,NaN,NaN,0
x4-doi-vai,3,9,9,1.0,0


bot_b2.json


,so_lan_chay,bot_bi_bao,tong_bot,ty_le_topology,cua_so_bao_nham
k4-bien-thien,3,9,9,1.0,0
k4-co-dinh,3,9,9,1.0,0
k4-doi-chung,3,NaN,NaN,NaN,0
k5-doi-chung,3,NaN,NaN,NaN,0
k5-p2p,3,9,9,0.987,0
k6-doi-chung,3,NaN,NaN,NaN,0
k6-p2p,3,15,15,0.989,0
k6-tap-trung,3,15,15,1.0,0
x4-doi-chung-doi-vai,3,NaN,NaN,NaN,0
x4-doi-vai,3,9,9,1.0,0
